# 🧪 PyTorch Lab13: Learning to Play Pong with Policy Gradients

This lab extends the previous Maze RL lab to a dynamic control problem: **Pong**.

We use a lightweight custom Pong environment instead of Atari Pong so that training remains feasible during a 3-hour lab. The agent observes an explicit low-dimensional state:

$$
s_t = [x_{ball}, y_{ball}, v^x_{ball}, v^y_{ball}, y_{agent}, y_{opponent}]
$$

The lab is divided into two 1.5-hour parts:

- **Part 1:** Pong environment + REINFORCE.
- **Part 2:** Actor-Critic with a learned value baseline, entropy regularization, advantage normalization, and gradient clipping.

## 0. Setup

We import the libraries required for the environment, neural networks, training curves, and GIF generation.

You do not need to modify the setup cells. Just run them.

In [ ]:
# If you run this notebook in a fresh Colab environment, uncomment the following lines.
# !pip -q install gymnasium imageio imageio-ffmpeg

import os
import math
import random
from collections import deque

import numpy as np
import gymnasium as gym
from gymnasium import spaces

import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib.pyplot as plt
import imageio
from IPython.display import Image, display
from tqdm.auto import tqdm

### Device

We use CUDA if it is available. Otherwise, the notebook runs on CPU.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

### Reproducibility helper

This function fixes the main random seeds. Reinforcement learning is still stochastic, but this makes results easier to compare.

In [ ]:
def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(42)

## 1. The SimplePong environment

This environment is intentionally simple and fully implemented for you.

The agent controls the **left paddle**. The opponent is a slow heuristic controller on the right side.

### State

The observation is a 6-dimensional vector:

$$
[x_{ball}, y_{ball}, v^x_{ball}, v^y_{ball}, y_{agent}, y_{opponent}]
$$

The velocity components are explicitly provided in the state. This keeps the lab focused on reinforcement learning rather than on inferring motion from stacked image frames.

### Actions

The action space is discrete:

- `0`: move paddle up
- `1`: move paddle down
- `2`: stay still

### Reward

The default training reward is shaped to make learning possible within a short lab:

- `+1.0` if the agent scores;
- `-1.0` if the opponent scores;
- `+0.5` by default if the agent hits the ball;
- `-0.001` step penalty at each time step;
- optional positive alignment reward when the ball approaches the agent.

When `reward_shaping=True` and the ball moves toward the agent, the environment adds:

$$
r_{	ext{align}} = \alpha (1 - |y_{	ext{agent}} - y_{	ext{ball}}|)
$$

where the default value is $\alpha = 0.02$.

This encourages the paddle to move close to the ball. During evaluation, reward shaping can be disabled with `reward_shaping=False`.

You do not need to modify this cell.

In [ ]:
class SimplePongEnv(gym.Env):
    metadata = {"render_modes": ["rgb_array"], "render_fps": 30}

    def __init__(
        self,
        render_mode="rgb_array",
        width=320,
        height=240,
        max_steps=800,
        reward_shaping=True,
        opponent_speed=0.018,
        hit_reward=0.5,
        tracking_reward_coef=0.02,
        step_penalty=-0.001,
        seed=None,
    ):
        super().__init__()
        self.render_mode = render_mode
        self.width = width
        self.height = height
        self.max_steps = max_steps
        self.reward_shaping = reward_shaping

        # Reward parameters.
        self.hit_reward = hit_reward
        self.tracking_reward_coef = tracking_reward_coef
        self.step_penalty = step_penalty

        # Continuous game geometry in [0, 1] x [0, 1].
        self.agent_x = 0.06
        self.opponent_x = 0.94
        self.paddle_w = 0.018
        self.paddle_h = 0.18
        self.ball_size = 0.025

        self.paddle_speed = 0.035
        self.opponent_speed = opponent_speed
        self.base_ball_speed_x = 0.018
        self.base_ball_speed_y = 0.014

        # 0=UP, 1=DOWN, 2=NOOP
        self.action_space = spaces.Discrete(3)

        # Velocity is normalized by 0.02 in the observation.
        low = np.array([0.0, 0.0, -2.0, -2.0, 0.0, 0.0], dtype=np.float32)
        high = np.array([1.0, 1.0, 2.0, 2.0, 1.0, 1.0], dtype=np.float32)
        self.observation_space = spaces.Box(low=low, high=high, dtype=np.float32)

        self.rng = np.random.default_rng(seed)
        self.reset(seed=seed)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None:
            self.rng = np.random.default_rng(seed)

        self.ball_x = 0.5
        self.ball_y = float(self.rng.uniform(0.35, 0.65))

        # Random initial direction.
        direction_x = float(self.rng.choice([-1.0, 1.0]))
        direction_y = float(self.rng.choice([-1.0, 1.0]))
        self.ball_vx = direction_x * self.base_ball_speed_x
        self.ball_vy = direction_y * float(self.rng.uniform(0.006, self.base_ball_speed_y))

        self.agent_y = 0.5
        self.opponent_y = 0.5
        self.steps = 0

        return self._get_obs(), {}

    def _get_obs(self):
        return np.array(
            [
                self.ball_x,
                self.ball_y,
                self.ball_vx / 0.02,
                self.ball_vy / 0.02,
                self.agent_y,
                self.opponent_y,
            ],
            dtype=np.float32,
        )

    def _clip_paddles(self):
        half = self.paddle_h / 2
        self.agent_y = float(np.clip(self.agent_y, half, 1.0 - half))
        self.opponent_y = float(np.clip(self.opponent_y, half, 1.0 - half))

    def _move_opponent(self):
        # A slow heuristic opponent: it follows the ball but cannot move instantly.
        if self.ball_y > self.opponent_y + 0.01:
            self.opponent_y += self.opponent_speed
        elif self.ball_y < self.opponent_y - 0.01:
            self.opponent_y -= self.opponent_speed

    def _paddle_collision(self, paddle_x, paddle_y, side):
        # side = "left" for the agent, "right" for the opponent.
        within_y = abs(self.ball_y - paddle_y) <= (self.paddle_h / 2 + self.ball_size / 2)

        if side == "left":
            within_x = self.ball_x <= paddle_x + self.paddle_w and self.ball_vx < 0
        else:
            within_x = self.ball_x >= paddle_x - self.paddle_w and self.ball_vx > 0

        return within_x and within_y

    def step(self, action):
        self.steps += 1
        reward = self.step_penalty
        terminated = False
        truncated = False

        # Agent movement.
        if action == 0:
            self.agent_y -= self.paddle_speed
        elif action == 1:
            self.agent_y += self.paddle_speed
        elif action == 2:
            pass
        else:
            raise ValueError(f"Invalid action: {action}")

        # Opponent movement.
        self._move_opponent()
        self._clip_paddles()

        # Ball movement.
        self.ball_x += self.ball_vx
        self.ball_y += self.ball_vy

        # Bounce on top and bottom walls.
        if self.ball_y <= self.ball_size / 2:
            self.ball_y = self.ball_size / 2
            self.ball_vy = abs(self.ball_vy)
        elif self.ball_y >= 1.0 - self.ball_size / 2:
            self.ball_y = 1.0 - self.ball_size / 2
            self.ball_vy = -abs(self.ball_vy)

        # Agent paddle collision.
        if self._paddle_collision(self.agent_x, self.agent_y, "left"):
            self.ball_x = self.agent_x + self.paddle_w
            self.ball_vx = abs(self.ball_vx) * 1.02
            relative_hit = (self.ball_y - self.agent_y) / (self.paddle_h / 2)
            self.ball_vy += 0.006 * relative_hit

            # Strong positive reward when the agent successfully returns the ball.
            reward += self.hit_reward

        # Opponent paddle collision.
        if self._paddle_collision(self.opponent_x, self.opponent_y, "right"):
            self.ball_x = self.opponent_x - self.paddle_w
            self.ball_vx = -abs(self.ball_vx) * 1.02
            relative_hit = (self.ball_y - self.opponent_y) / (self.paddle_h / 2)
            self.ball_vy += 0.006 * relative_hit

        # Limit ball speed for numerical stability.
        self.ball_vy = float(np.clip(self.ball_vy, -0.03, 0.03))
        self.ball_vx = float(np.clip(self.ball_vx, -0.03, 0.03))

        # Optional reward shaping:
        # When the ball moves toward the agent, encourage vertical alignment.
        if self.reward_shaping and self.ball_vx < 0:
            distance = abs(self.agent_y - self.ball_y)
            alignment_score = 1.0 - distance
            reward += self.tracking_reward_coef * alignment_score

        # Scoring.
        # The agent loses only when the ball crosses the left boundary.
        if self.ball_x < 0.0:
            reward += -1.0
            terminated = True

        # The agent scores only when the ball crosses the right boundary.
        elif self.ball_x > 1.0:
            reward += 1.0
            terminated = True

        if self.steps >= self.max_steps:
            truncated = True

        return self._get_obs(), float(reward), terminated, truncated, {}

    def render(self):
        canvas = np.zeros((self.height, self.width, 3), dtype=np.uint8)
        canvas[:, :, :] = 10

        # Center line.
        center_x = self.width // 2
        canvas[::12, center_x - 1:center_x + 1, :] = 120

        def rect(cx, cy, w, h, color):
            x0 = int((cx - w / 2) * self.width)
            x1 = int((cx + w / 2) * self.width)
            y0 = int((cy - h / 2) * self.height)
            y1 = int((cy + h / 2) * self.height)
            x0, x1 = np.clip([x0, x1], 0, self.width - 1)
            y0, y1 = np.clip([y0, y1], 0, self.height - 1)
            canvas[y0:y1, x0:x1, :] = color

        # Paddles and ball.
        rect(self.agent_x, self.agent_y, self.paddle_w, self.paddle_h, (230, 230, 230))
        rect(self.opponent_x, self.opponent_y, self.paddle_w, self.paddle_h, (180, 180, 180))
        rect(self.ball_x, self.ball_y, self.ball_size, self.ball_size, (255, 255, 255))

        return canvas

## 2. Pong as a Markov Decision Process

Before training, we explicitly map the environment to an MDP.

- **State:** ball position, ball velocity, agent paddle position, opponent paddle position.
- **Action:** move up, move down, or stay.
- **Reward:** positive for scoring or hitting the ball, negative for conceding a point or being far from the ball.
- **Policy:** a neural network that outputs a probability distribution over actions.
- **Episode:** one rally, ending when one side scores or when the maximum number of steps is reached.

The important difference with Maze is that Pong is dynamic: the ball continues moving even if the agent does nothing.

In [ ]:
action_meanings = {0: "UP", 1: "DOWN", 2: "NOOP"}
print("Action meanings:", action_meanings)

example_env = SimplePongEnv(seed=0)
obs, info = example_env.reset()
print("Observation shape:", obs.shape)
print("Initial observation:", obs)
print("Observation meaning: [ball_x, ball_y, ball_vx, ball_vy, agent_y, opponent_y]")

## 3. Visual sanity check

At this point, the environment is already complete. There is no TODO before or inside this section.

Run the following cells to verify that:

1. the environment resets correctly;
2. a random policy can interact with it;
3. rendering works;
4. the ball and paddles are visible.

In [ ]:
env = SimplePongEnv(seed=1)
obs, info = env.reset()

frames = []
total_reward = 0.0
for t in range(120):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    frames.append(env.render())
    if terminated or truncated:
        break

print("Number of frames:", len(frames))
print("Total reward from random policy:", round(total_reward, 3))
plt.figure(figsize=(5, 4))
plt.imshow(frames[-1])
plt.axis("off")
plt.title("Last frame from a random-policy rollout")
plt.show()

In [ ]:
def save_gif(frames, filename="pong_random.gif", fps=30):
    imageio.mimsave(filename, frames, fps=fps)
    return filename

gif_path = save_gif(frames, "pong_random.gif")
display(Image(filename=gif_path))

## 4. Utility functions

From this section onward, students will complete TODO blocks.

The first utility function computes discounted returns:

$$
G_t = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \cdots
$$

This is the Monte Carlo learning signal used by REINFORCE and by the critic target in Part 2.

In [ ]:
def discounted_returns(rewards, gamma=0.99):
    """
    Compute discounted returns G_t for one episode.

    Args:
        rewards: list of rewards [r_0, r_1, ..., r_{T-1}]
        gamma: discount factor

    Returns:
        torch.FloatTensor of shape [T]
    """
    returns = []
    G = 0.0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    return torch.tensor(returns, dtype=torch.float32, device=device)

### Quick check for discounted returns

For rewards `[1, 1, 1]` and `gamma = 0.9`, the returns should be approximately `[2.71, 1.9, 1.0]`.

In [ ]:
# Run this after completing discounted_returns.
# Expected output: tensor([2.7100, 1.9000, 1.0000])
print(discounted_returns([1, 1, 1], gamma=0.9))

## Part 1 — REINFORCE

## 5. Policy network

The policy network receives the 6-dimensional state and outputs 3 logits, one for each action.

We will sample actions from:

$$
\pi_\theta(a_t \mid s_t)
$$

using `torch.distributions.Categorical`.

In [ ]:
class PolicyNet(nn.Module):
    def __init__(self, obs_dim=6, n_actions=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, n_actions),
        )

    def forward(self, x):
        return self.net(x)

## 6. Selecting an action

Given an observation, the policy network outputs logits. We then construct a categorical distribution and sample one action.

The function returns:

- the sampled action as an integer;
- the log-probability of that action;
- the entropy of the action distribution.

Entropy is useful for diagnostics and for Part 2.

In [ ]:
def select_action(policy, obs):
    """
    Select an action using the current policy.
    """
    obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
    logits = policy(obs_t)
    dist = torch.distributions.Categorical(logits=logits)
    action = dist.sample()
    log_prob = dist.log_prob(action)
    entropy = dist.entropy()
    return int(action.item()), log_prob.squeeze(0), entropy.squeeze(0)

## 7. Collecting one episode

A policy-gradient method learns from trajectories.

This function runs one full episode and stores:

- states;
- actions;
- rewards;
- log-probabilities;
- entropies.

The stored log-probabilities are used in the REINFORCE loss.

In [ ]:
def run_episode(env, policy, max_steps=None):
    obs, info = env.reset()

    states = []
    actions = []
    rewards = []
    log_probs = []
    entropies = []
    frames = []

    done = False
    steps = 0

    while not done:
        action, log_prob, entropy = select_action(policy, obs)
        next_obs, reward, terminated, truncated, info = env.step(action)

        states.append(obs)
        actions.append(action)
        rewards.append(reward)
        log_probs.append(log_prob)
        entropies.append(entropy)
        frames.append(env.render())

        obs = next_obs
        done = terminated or truncated
        steps += 1

        if max_steps is not None and steps >= max_steps:
            break

    return {
        "states": np.array(states, dtype=np.float32),
        "actions": np.array(actions, dtype=np.int64),
        "rewards": rewards,
        "log_probs": log_probs,
        "entropies": entropies,
        "frames": frames,
        "return": float(np.sum(rewards)),
    }

## 8. REINFORCE update

The REINFORCE objective is to maximize expected return. In code, we minimize the negative objective:

$$
\mathcal{L}_{actor} = -\frac{1}{T}\sum_t \log \pi_\theta(a_t \mid s_t) G_t
$$

We normalize the returns to reduce variance and make optimization more stable.

In [ ]:
def update_reinforce(policy, optimizer, episodes, gamma=0.99):
    all_log_probs = []
    all_returns = []
    all_entropies = []

    for ep in episodes:
        returns = discounted_returns(ep["rewards"], gamma=gamma)
        all_log_probs.extend(ep["log_probs"])
        all_returns.append(returns)
        all_entropies.extend(ep["entropies"])

    log_probs = torch.stack(all_log_probs)
    returns = torch.cat(all_returns)
    entropies = torch.stack(all_entropies)

    returns = (returns - returns.mean()) / (returns.std() + 1e-8)
    actor_loss = -(log_probs * returns).mean()

    optimizer.zero_grad()
    actor_loss.backward()
    nn.utils.clip_grad_norm_(policy.parameters(), max_norm=0.5)
    optimizer.step()

    return {
        "actor_loss": float(actor_loss.item()),
        "mean_entropy": float(entropies.mean().item()),
        "mean_return_target": float(returns.mean().item()),
    }

## 9. Training REINFORCE

We collect several episodes, update the policy, and repeat.

For a classroom session, start with a small number of episodes. You can increase it if time allows.

In [ ]:
def moving_average(x, window=25):
    if len(x) < window:
        return np.array(x)
    return np.convolve(x, np.ones(window) / window, mode="valid")

set_seed(42)
env = SimplePongEnv(seed=42, reward_shaping=True)
policy = PolicyNet().to(device)
optimizer = torch.optim.Adam(policy.parameters(), lr=5e-4)

N_EPISODES = 300
BATCH_EPISODES = 10
GAMMA = 0.99

reinforce_returns = []
reinforce_losses = []
reinforce_entropies = []

for start_ep in tqdm(range(0, N_EPISODES, BATCH_EPISODES)):
    episodes = [run_episode(env, policy) for _ in range(BATCH_EPISODES)]
    stats = update_reinforce(policy, optimizer, episodes, gamma=GAMMA)

    reinforce_returns.extend([ep["return"] for ep in episodes])
    reinforce_losses.append(stats["actor_loss"])
    reinforce_entropies.append(stats["mean_entropy"])

print("Final average return over last 20 episodes:", np.mean(reinforce_returns[-20:]))

## 10. Plotting REINFORCE results

The moving average smooths the return curve and makes the learning trend easier to see.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(reinforce_returns, alpha=0.35, label="Episode return")
ma = moving_average(reinforce_returns, window=25)
plt.plot(range(len(ma)), ma, label="Moving average (25)")
plt.xlabel("Episode")
plt.ylabel("Return")
plt.title("REINFORCE on SimplePong")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(reinforce_entropies)
plt.xlabel("Update")
plt.ylabel("Mean policy entropy")
plt.title("Policy entropy during REINFORCE training")
plt.grid(True)
plt.show()

## 10.1 Evaluation and GIF helpers

The following helpers evaluate a trained policy greedily and record a GIF.

For `PolicyNet`, the forward pass returns only logits. For `ActorCriticNet`, the forward pass returns `(logits, value)`. The helper below supports both.

In [ ]:
def get_logits(model, obs):
    obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
    with torch.no_grad():
        output = model(obs_t)
        if isinstance(output, tuple):
            logits = output[0]
        else:
            logits = output
    return logits


def greedy_action(model, obs):
    logits = get_logits(model, obs)
    return int(torch.argmax(logits, dim=-1).item())


def evaluate_policy(env, model, n_episodes=20):
    returns = []
    for _ in range(n_episodes):
        obs, info = env.reset()
        done = False
        total = 0.0
        while not done:
            action = greedy_action(model, obs)
            obs, reward, terminated, truncated, info = env.step(action)
            total += reward
            done = terminated or truncated
        returns.append(total)
    return float(np.mean(returns)), float(np.std(returns))


def record_policy_gif(env, model, filename="pong_trained_agent.gif", fps=30, max_steps=800):
    obs, info = env.reset()
    frames = []
    done = False
    steps = 0
    total = 0.0

    while not done and steps < max_steps:
        action = greedy_action(model, obs)
        obs, reward, terminated, truncated, info = env.step(action)
        total += reward
        frames.append(env.render())
        done = terminated or truncated
        steps += 1

    imageio.mimsave(filename, frames, fps=fps)
    print(f"Saved {filename} with return {total:.3f} and {len(frames)} frames.")
    return filename

## 10.2 Evaluate and visualize the REINFORCE agent

REINFORCE may still behave poorly after a few hundred episodes. This is expected on Pong because the method has high variance and no learned baseline.

We nevertheless record a GIF so that we can visually compare REINFORCE with Actor-Critic later.

In [ ]:
reinforce_eval_env = SimplePongEnv(seed=999, reward_shaping=False)
mean_return, std_return = evaluate_policy(reinforce_eval_env, policy, n_episodes=20)
print(f"REINFORCE greedy evaluation: mean={mean_return:.3f}, std={std_return:.3f}")

gif_path = record_policy_gif(SimplePongEnv(seed=1000, reward_shaping=False), policy, "pong_reinforce.gif")
display(Image(filename=gif_path))

# Part 2 — Actor-Critic with a learned baseline

REINFORCE has high variance because actions are weighted directly by Monte Carlo returns.

We now learn a value function:

$$
V_\phi(s_t) \approx \mathbb{E}[G_t \mid s_t]
$$

and use the advantage estimate:

$$
\hat{A}_t = G_t - V_\phi(s_t)
$$

The actor is updated with the advantage, and the critic is trained by regression on returns.

## 11. Actor-Critic network

The network has:

- a shared feature extractor;
- a policy head that outputs action logits;
- a value head that outputs one scalar value per state.

In [ ]:
class ActorCriticNet(nn.Module):
    def __init__(self, obs_dim=6, n_actions=3):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(obs_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
        )
        self.policy_head = nn.Linear(64, n_actions)
        self.value_head = nn.Linear(64, 1)

    def forward(self, x):
        z = self.trunk(x)
        logits = self.policy_head(z)
        value = self.value_head(z).squeeze(-1)
        return logits, value

## 12. Selecting actions with Actor-Critic

Action selection uses only the policy head. The value head is used later during the update.

In [ ]:
def select_action_ac(model, obs):
    obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
    logits, value = model(obs_t)
    dist = torch.distributions.Categorical(logits=logits)
    action = dist.sample()
    log_prob = dist.log_prob(action)
    entropy = dist.entropy()
    return int(action.item()), log_prob.squeeze(0), entropy.squeeze(0), value.squeeze(0)

## 13. Collecting one Actor-Critic episode

This version also stores value estimates produced during interaction.

In [ ]:
def run_episode_ac(env, model, max_steps=None):
    obs, info = env.reset()

    states = []
    actions = []
    rewards = []
    log_probs = []
    entropies = []
    values = []
    frames = []

    done = False
    steps = 0

    while not done:
        action, log_prob, entropy, value = select_action_ac(model, obs)
        next_obs, reward, terminated, truncated, info = env.step(action)

        states.append(obs)
        actions.append(action)
        rewards.append(reward)
        log_probs.append(log_prob)
        entropies.append(entropy)
        values.append(value)
        frames.append(env.render())

        obs = next_obs
        done = terminated or truncated
        steps += 1

        if max_steps is not None and steps >= max_steps:
            break

    return {
        "states": np.array(states, dtype=np.float32),
        "actions": np.array(actions, dtype=np.int64),
        "rewards": rewards,
        "log_probs": log_probs,
        "entropies": entropies,
        "values": values,
        "frames": frames,
        "return": float(np.sum(rewards)),
    }

## 14. Actor-Critic update

The total loss has three terms:

$$
\mathcal{L} = \mathcal{L}_{actor} + c_v \mathcal{L}_{critic} - \beta H(\pi)
$$

where:

$$
\mathcal{L}_{actor} = -\log \pi_\theta(a_t \mid s_t) \hat{A}_t
$$

$$
\mathcal{L}_{critic} = (V_\phi(s_t) - G_t)^2
$$

and entropy encourages exploration.

In [ ]:
def update_actor_critic(
    model,
    optimizer,
    episodes,
    gamma=0.99,
    value_coef=0.5,
    entropy_coef=0.01,
    max_grad_norm=0.5,
):
    all_log_probs = []
    all_returns = []
    all_values = []
    all_entropies = []

    for ep in episodes:
        returns = discounted_returns(ep["rewards"], gamma=gamma)
        all_log_probs.extend(ep["log_probs"])
        all_returns.append(returns)
        all_values.extend(ep["values"])
        all_entropies.extend(ep["entropies"])

    log_probs = torch.stack(all_log_probs)
    returns = torch.cat(all_returns)
    values = torch.stack(all_values)
    entropies = torch.stack(all_entropies)

    advantages = returns - values.detach()
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    actor_loss = -(log_probs * advantages).mean()
    critic_loss = F.mse_loss(values, returns)
    entropy_bonus = entropies.mean()

    loss = actor_loss + value_coef * critic_loss - entropy_coef * entropy_bonus

    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
    optimizer.step()

    return {
        "loss": float(loss.item()),
        "actor_loss": float(actor_loss.item()),
        "critic_loss": float(critic_loss.item()),
        "mean_entropy": float(entropies.mean().item()),
        "mean_value": float(values.mean().item()),
    }

## 15. Training Actor-Critic

We train with the same environment and compare learning curves with REINFORCE.

In [ ]:
set_seed(123)
env = SimplePongEnv(seed=123, reward_shaping=True)
ac_model = ActorCriticNet().to(device)
ac_optimizer = torch.optim.Adam(ac_model.parameters(), lr=1e-3)

N_EPISODES = 250
BATCH_EPISODES = 5
GAMMA = 0.99
VALUE_COEF = 0.5
ENTROPY_COEF = 0.01

ac_returns = []
ac_actor_losses = []
ac_critic_losses = []
ac_entropies = []

for start_ep in tqdm(range(0, N_EPISODES, BATCH_EPISODES)):
    episodes = [run_episode_ac(env, ac_model) for _ in range(BATCH_EPISODES)]
    stats = update_actor_critic(
        ac_model,
        ac_optimizer,
        episodes,
        gamma=GAMMA,
        value_coef=VALUE_COEF,
        entropy_coef=ENTROPY_COEF,
    )

    ac_returns.extend([ep["return"] for ep in episodes])
    ac_actor_losses.append(stats["actor_loss"])
    ac_critic_losses.append(stats["critic_loss"])
    ac_entropies.append(stats["mean_entropy"])

print("Final average return over last 20 episodes:", np.mean(ac_returns[-20:]))

## 16. Comparing REINFORCE and Actor-Critic

Compare the moving average returns. Actor-Critic is not guaranteed to win in every short run, but it should usually be more stable because the value function reduces variance.

In [ ]:
plt.figure(figsize=(9, 4))
reinforce_ma = moving_average(reinforce_returns, window=25)
ac_ma = moving_average(ac_returns, window=25)
plt.plot(range(len(reinforce_ma)), reinforce_ma, label="REINFORCE")
plt.plot(range(len(ac_ma)), ac_ma, label="Actor-Critic")
plt.xlabel("Episode")
plt.ylabel("Moving average return")
plt.title("REINFORCE vs Actor-Critic on SimplePong")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(ac_actor_losses, label="Actor loss")
plt.plot(ac_critic_losses, label="Critic loss")
plt.xlabel("Update")
plt.title("Actor-Critic losses")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(ac_entropies, label="Actor-Critic entropy")
plt.plot(reinforce_entropies, label="REINFORCE entropy", alpha=0.7)
plt.xlabel("Update")
plt.ylabel("Mean entropy")
plt.title("Policy entropy")
plt.legend()
plt.grid(True)
plt.show()

## 17. Evaluation and GIF generation

We now evaluate and visualize the final Actor-Critic agent using the same greedy evaluation helpers as before.

In [ ]:
eval_env = SimplePongEnv(seed=999, reward_shaping=False)
mean_return, std_return = evaluate_policy(eval_env, ac_model, n_episodes=20)
print(f"Actor-Critic greedy evaluation: mean={mean_return:.3f}, std={std_return:.3f}")

gif_path = record_policy_gif(SimplePongEnv(seed=1000, reward_shaping=False), ac_model, "pong_actor_critic.gif")
display(Image(filename=gif_path))

## 18. Questions

Answer the following questions after completing the lab.

1. Why is ball velocity important in Pong?
2. Why is Pong more difficult than Maze?
3. What does the value function learn?
4. Why do we subtract $V(s_t)$ from $G_t$?
5. What is the role of entropy regularization?
6. What changes when reward shaping is disabled during evaluation?

## Optional extensions

1. Remove `ball_vx` and `ball_vy` from the state and stack the last 4 low-dimensional observations instead.
2. Disable reward shaping during training and compare the learning curve.
3. Increase the opponent speed and observe whether the trained agent still performs well.
4. Replace the low-dimensional state with image frames and train a CNN policy.